# Introduction

This notebook contains the code for the simulation environment used in the project. It defines the classes and functions needed to simulate the operation of a battery storage system in an electricity market, including the state representation, action space, exogenous data generation, and simulation loop. 

The environment is designed to be flexible and extensible, allowing for different policies and exogenous data generators to be easily integrated.



## Importing libraries

The following libraries are imported for use in the simulation environment:

In [ ]:
pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
from pyomo.environ import *
from copy import deepcopy
import matplotlib.pyplot as plt
import pickle
import torch
import pickle
import torch.nn as nn

# Simulation Environment

We are going to create a simulation environment to model the energy market for a single consumer. 
The consumer is small enough to not affect the market price, as this would otherwise make the problem more complex. 
It takes the market price $p_t$ as given.

We also assume that the consumer knows their own energy consumption patterns in advance and it is $d_t$ for each time period $t$. 
For many consumers, this is a reasonable assumption, as much of energy consumption is predictable (e.g., heating, cooling, appliances). 
Some consumers may have more variable consumption patterns, but this would require a more complex model to capture the uncertainty in demand.

We assume that the demand must be met at all times, meaning that the consumer cannot simply choose to not consume energy during peak hours.
This is a reasonable assumption, as most consumers have essential energy needs that cannot be easily shifted (e.g., heating, cooling, lighting), and if they could be shifted, you could simply shift them to off-peak hours.

The consumer buys $x_t$ kWh of energy from the market at time $t$.

The consumer has a battery of size $b_{max}$ kWh.

The battery has a maximum charge rate of $b_{charge\_max}$ kW and a discharge rate of $b_{discharge\_max}$ kW.

The battery has an charging efficiency of $b_{charge\_efficiency}$, meaning that if you charge the battery with 1 kWh, only get $b_{charge\_efficiency}$ kWh gets stored.

The battery has a discharging efficiency of $b_{discharge\_efficiency}$, meaning that if you discharge the battery with 1 kWh, only get $b_{discharge\_efficiency}$ kWh of usable energy.

Other battery factors that could be considered, but are not included in the current model, include:

Degradation of the battery, as it may affect the long-term cost-effectiveness of the battery.

Passive energy losses, as the battery may lose energy over time even when not in use.


## Nomenclature

### Sets
$T$: Set of time periods (hours) indexed by $t$.

### Parameters
$p_t$: Buying price of electricity at time $t$ (EUR/kWh).

$c_t$: Carbon intensity of electricity at time $t$ (kgCO2/kWh).

$c_{price}$: Carbon price (EUR/kgCO2).

$d_t$: Energy demand at time $t$ (kWh).

$b_{max}$: Maximum battery capacity (kWh).

$b_{charge\_max}$: Maximum battery charge rate (kW).

$b_{charge\_efficiency}$: Battery charging efficiency (0 < $b_{charge\_efficiency}$ ≤ 1).

$b_{discharge\_max}$: Maximum battery discharge rate (kW).

$b_{discharge\_efficiency}$: Battery discharging efficiency (0 < $b_{discharge\_efficiency}$ ≤ 1).


### Variables
$x_t$: Amount of electricity purchased at time $t$ (kWh).

$b_t$: Battery level at time $t$ (kWh).



## State

The state consists of the following components:

Battery level, $b_t$

Buying price of electricity, $p_t$

Carbon intensity of electricity, $c_t$

Other factors will be included in the state to help the agent predict future states like weather conditions or economic indicators.


In [ ]:
class State:
    def __init__(self, time_step, battery_level, energy_price, carbon_intensity, other_factors=None):
        self.time_step = time_step
        self.battery_level = battery_level
        self.energy_price = energy_price
        self.carbon_intensity = carbon_intensity
        self.other_factors = other_factors



## Action
Each hour decide how much energy to buy, $x_t$.

If $x_t$ is bigger than demand, the excess energy is stored in the battery.

If $x_t$ is smaller than demand, the deficit is covered by the battery.


In [ ]:
class Action:
    def __init__(self, buy_energy_amount):
        self.buy_energy_amount = buy_energy_amount


## Reward (cost)

The total cost for each action $a_t$ at state $s_t$ is the cost of buying energy from the market plus the cost of the carbon emissions.

$$c(s_t, a_t) = (p_t + c_{price} \cdot c_t) \cdot x_t$$


In [ ]:
class Cost: 
    def __init__(self, amount, carbon_emissions, price_per_kg_co2):
        self.amount = amount
        self.carbon_emissions = carbon_emissions
        self.total_cost = amount + price_per_kg_co2 * carbon_emissions



## Parameters

The parameters of the environment are included in its own class, which allows for easy modification and experimentation with different parameter values.

The parameters are given to the environment at initialization.

In [ ]:
class Parameters:
    def __init__(self, initial_battery_level, max_battery_capacity, max_charging_rate, max_discharging_rate, charge_efficiency, discharge_efficiency, price_per_kg_co2):
        self.initial_battery_level = initial_battery_level
        self.max_battery_capacity = max_battery_capacity
        self.max_charging_rate = max_charging_rate
        self.max_discharging_rate = max_discharging_rate
        self.charge_efficiency = charge_efficiency
        self.discharge_efficiency = discharge_efficiency
        self.price_per_kg_co2 = price_per_kg_co2

## The exogenous data

The exogenous data for each time period $t$ includes the market price of electricity $p_t$, the carbon intensity of electricity $c_t$, and some other factors that can help the agent predict future market prices like weather conditions or economic indicators.

A exgogenous trajectory is defined as a sequence of exogenous data for each time period .

In [ ]:

class ExogenousValues:
    def __init__(self, energy_price, carbon_intensity, other_factors=None):
        self.energy_price = energy_price
        self.carbon_intensity = carbon_intensity
        self.other_factors = other_factors

class ExogenousTrajectory:
    def __init__(self, zone, exogen_values_list, demand_list):
        self.zone = zone
        self.exogen_values_list = exogen_values_list
        self.demand_list = demand_list



## Simulation results classes

When running a simulation, we want to store the results in a structured way.


In [ ]:
class SimulationResult:
    def __init__(self, exogen_trajectory=None):
        self.exogen_trajectory = exogen_trajectory
        self.states = []
        self.actions = []
        self.costs = []


    def get_total_cost(self, cost_type='total_cost'):
        return sum(getattr(cost, cost_type) for cost in self.costs)
    
    def is_similar_to(self, other_simulation_result, tolerance=1e-6):
        for t in range(len(self.states)):
            assert abs(self.states[t].energy_price - self.exogen_trajectory.exogen_values_list[t].energy_price) <= tolerance, f"States differ at time step {t}: {self.states[t].energy_price} vs {self.exogen_trajectory.exogen_values_list[t].energy_price}"
            assert abs(self.states[t].carbon_intensity - self.exogen_trajectory.exogen_values_list[t].carbon_intensity) <= tolerance, f"States differ at time step {t}: {self.states[t].carbon_intensity} vs {self.exogen_trajectory.exogen_values_list[t].carbon_intensity}"

        for t in range(len(self.states)):
            assert abs(self.states[t].battery_level - other_simulation_result.states[t].battery_level) <= tolerance, f"States differ at time step {t}: {self.states[t].battery_level} vs {other_simulation_result.states[t].battery_level}"
            assert abs(self.states[t].energy_price - other_simulation_result.states[t].energy_price) <= tolerance, f"States differ at time step {t}: {self.states[t].energy_price} vs {other_simulation_result.states[t].energy_price}"
            #assert abs(self.states[t].carbon_intensity - other_simulation_result.states[t].carbon_intensity) <= tolerance, f"States differ at time step {t}: {self.states[t].carbon_intensity} vs {other_simulation_result.states[t].carbon_intensity}"

        for t in range(len(self.actions)):
            assert abs(self.actions[t].buy_energy_amount - other_simulation_result.actions[t].buy_energy_amount) <= tolerance, f"Actions differ at time step {t}: {self.actions[t].buy_energy_amount} vs {other_simulation_result.actions[t].buy_energy_amount}"

        for t in range(len(self.costs)):
            assert abs(self.costs[t].amount - other_simulation_result.costs[t].amount) <= tolerance, f"Costs differ at time step {t}: {self.costs[t].amount} vs {other_simulation_result.costs[t].amount}"
            #assert abs(self.costs[t].carbon_emissions - other_simulation_result.costs[t].carbon_emissions) <= tolerance, f"Costs differ at time step {t}: {self.costs[t].carbon_emissions} vs {other_simulation_result.costs[t].carbon_emissions}"
            #assert abs(self.costs[t].total_cost - other_simulation_result.costs[t].total_cost) <= tolerance, f"Costs differ at time step {t}: {self.costs[t].total_cost} vs {other_simulation_result.costs[t].total_cost}"
            
    def plot(self, environment, start=0, end=None):
        # price
        plt.figure(figsize=(14, 16))
        plt.subplot(6, 1, 1)
        plt.title(f"Simulation Result for Exogenous Trajectory: {self.exogen_trajectory.zone} (start={start}, end={end})")
        plt.plot(range(start, end), [state.energy_price for state in self.states[start:end]], label='Energy Price (EUR/kWh)')
        plt.ylabel('Energy Price (EUR/kWh)')
        plt.legend()
        # carbon intensity
        plt.subplot(6, 1, 2)
        plt.plot(range(start, end), [state.carbon_intensity for state in self.states[start:end]], label='Carbon Intensity (kgCO2/kWh)', color='orange')
        plt.ylabel('Carbon Intensity (kgCO2/kWh)')
        plt.legend()
        # demand 
        plt.subplot(6, 1, 3)
        plt.plot(range(start, end), [environment.demand_profile[state.time_step] for state in self.states[start:end]], label='Demand (kWh)', color='blue')
        plt.ylabel('Demand (kWh)')
        plt.legend()
        # buy energy amount
        plt.subplot(6, 1, 4)
        plt.plot(range(start, end), [action.buy_energy_amount for action in self.actions[start:end]], label='Energy Bought (kWh)', color='red')
        plt.ylabel('Energy Bought (kWh)')
        #plt.xlabel('Time Step')
        plt.legend()
        # excess energy (positive for charging, negative for discharging)
        plt.subplot(6, 1, 5)
        plt.plot(range(start, end), [action.buy_energy_amount - environment.demand_profile[state.time_step] for state, action in zip(self.states[start:end], self.actions[start:end])], label='Excess Energy (kWh)', color='purple')
        plt.ylabel('Excess Energy (kWh)')
        #plt.xlabel('Time Step')
        plt.legend()
        # battery level
        plt.subplot(6, 1, 6)
        plt.plot(range(start, end), [state.battery_level for state in self.states[start:end]], label='Battery Level (kWh)', color='green')
        plt.ylabel('Battery Level (kWh)')
        #plt.xlabel('Time Step')
        plt.legend()
        plt.show()

class EvaluationResult:
    def __init__(self, policy_name):
        self.policy_name = policy_name
        self.simulation_results = []

    def print_average_costs(self):
        for cost_type in ['amount', 'carbon_emissions', 'total_cost']:
            avg_cost = sum(sim.get_total_cost(cost_type) for sim in self.simulation_results) / len(self.simulation_results)
            print(f"Average {cost_type}: {avg_cost:.2f}")
        


## Environment

## Feasibility
$0 \leq b_t \leq b_{max}$ (battery capacity constraint)
$x_t - d_t \leq b_{charge\_max}$ (battery charge rate constraint)
$d_t - x_t \leq b_{discharge\_max}$ (battery discharge rate constraint)


## Dynamics
If $x_t > d_t$, then the excess energy is stored in the battery, and the battery level increases by $(x_t - d_t) * b_{charge\_efficiency}\%$.
$$b_{t+1} = b_t + (x_t - d_t) * b_{charge\_efficiency}\% $$
If $x_t < d_t$, then the deficit is covered by the battery, and the battery level decreases by $d_t - x_t$.
$$b_{t+1} = b_t - (d_t - x_t)$$

The other factors market prices, unpredictable demand, weather conditions, and economic indicators are exogenous and can be modeled as stochastic processes or based on historical data. 


In [ ]:
class Environment:
    def __init__(self, parameters, epsilon=1e-4):
        self.parameters = parameters
        self.epsilon = epsilon
        self.demand_profile = None  # This will be set for each simulation based on the exogenous trajectory being used


    def check_and_fix_action(self, state, action):
        # return True if action is feasible in the given state, else False

        if action.buy_energy_amount > self.demand_profile[state.time_step]:  # Charging
            lowest_max_limit = min(self.demand_profile[state.time_step] + self.parameters.max_charging_rate,
                               self.demand_profile[state.time_step] + (self.parameters.max_battery_capacity - state.battery_level) / self.parameters.charge_efficiency)
            
            if action.buy_energy_amount > lowest_max_limit + self.epsilon:
                print("Action exceeds limits for charging.")
                print(f"Action buy energy amount: {action.buy_energy_amount} kWh")
                print(f"Demand at time step {state.time_step}: {self.demand_profile[state.time_step]} kWh")
                print(f"Lowest limit for charging: {lowest_max_limit} kWh")
                print(f"Battery level: {state.battery_level} kWh")
                
                raise ValueError("Action exceeds limits for charging.")
            elif action.buy_energy_amount > lowest_max_limit:
                action.buy_energy_amount = lowest_max_limit

        else:  # Discharging
            highest_min_limit = max(self.demand_profile[state.time_step] - self.parameters.max_discharging_rate * self.parameters.discharge_efficiency,
                                    self.demand_profile[state.time_step] - state.battery_level * self.parameters.discharge_efficiency)


            if  action.buy_energy_amount < highest_min_limit - self.epsilon:
                print("Action exceeds limits for discharging.")
                print(f"Action buy energy amount: {action.buy_energy_amount} kWh")
                print(f"Demand at time step {state.time_step}: {self.demand_profile[state.time_step]} kWh")
                print(f"Highest limit for discharging: {highest_min_limit} kWh")
                print(f"Battery level: {state.battery_level} kWh")
                
                raise ValueError("Action exceeds limits for discharging.")
            elif action.buy_energy_amount < highest_min_limit:
                action.buy_energy_amount = highest_min_limit

        return action

    def get_cost(self, state, action):
        # return cost of taking action in the given state
        energy_cost_amount = action.buy_energy_amount * state.energy_price
        carbon_emissions = action.buy_energy_amount * state.carbon_intensity
        return Cost(amount=energy_cost_amount, carbon_emissions=carbon_emissions, price_per_kg_co2=self.parameters.price_per_kg_co2)

    def transition(self, state, action, next_exogen_values):
        # return next state
        excess_energy = action.buy_energy_amount - self.demand_profile[state.time_step]

        # update battery level based on action
        if excess_energy > 0:  # Charging
            new_battery_level = state.battery_level + excess_energy * self.parameters.charge_efficiency
        else:  # Discharging
            new_battery_level = state.battery_level + excess_energy / self.parameters.discharge_efficiency  # excess_energy is negative for discharging

        # get next exogen values
        return State(time_step=state.time_step + 1,
                     battery_level=new_battery_level,
                     energy_price=next_exogen_values.energy_price,
                     carbon_intensity=next_exogen_values.carbon_intensity,
                     other_factors=next_exogen_values.other_factors)
    
    def do_simulation(self, policy, exogen_trajectory):
        self.demand_profile = exogen_trajectory.demand_list
        exogen_data = exogen_trajectory.exogen_values_list

        if policy.is_hindsight_policy:
            policy.see_future_exogen_values(exogen_data)

        result = SimulationResult(exogen_trajectory)

        state = State(time_step=0,
                        battery_level=self.parameters.initial_battery_level,
                        energy_price=exogen_data[0].energy_price,
                        carbon_intensity=exogen_data[0].carbon_intensity,
                        other_factors=exogen_data[0].other_factors)

        for t in range(len(exogen_data)):
            result.states.append(state)

            policy_action = policy.select_action(state)
            policy_action = self.check_and_fix_action(state, policy_action)
            
            result.actions.append(policy_action)
            
            
            cost = self.get_cost(state, policy_action)
            result.costs.append(cost)

            if t < len(exogen_data) - 1:  # No need to transition on the last step
                next_exogen_values = exogen_data[t + 1]
                state = self.transition(state, policy_action, next_exogen_values)

        if policy.expected_simulation_result is not None:
            # check that hindsight policy actions are the same as optimal actions
            result.is_similar_to(policy.expected_simulation_result, tolerance=self.epsilon)

        return result
        

    def evaluate_policies(self, policies, all_exogen_trajectories):

        all_results = []
        for policy in policies:
            policy_name = policy.__name__
            print(f"Evaluating policy: {policy_name}")
            result = EvaluationResult(policy_name)
        
            for sim, exogen_trajectory in enumerate(all_exogen_trajectories):
                print(f"  Simulation {sim + 1}/{len(all_exogen_trajectories)} ({exogen_trajectory.zone})")
                
                policy_instance = policy(self)
                simulation_result = self.do_simulation(policy_instance, exogen_trajectory)
                result.simulation_results.append(simulation_result)

                
            print(f"Policy {policy_name} evaluation completed.")
            result.print_average_costs()
            
            all_results.append(result)

        return all_results


    

## Agent
Represents a consumer who wants to minimize their energy costs by deciding how much energy to buy from the market.

Its primary method is select_action(state), which takes the current state as input and returns the action (amount of energy to buy) that the agent decides to take.

In [ ]:
class Agent:
    def __init__(self, environment):
        self.environment = environment
        self.is_hindsight_policy = False  # Set to True for hindsight agents that can see future exogenous values
        self.expected_simulation_result = None  # For hindsight agents to store the expected optimal simulation result based on the exogenous data they see

    def see_future_exogen_values(self, exogen_data):
        raise NotImplementedError("This method should be implemented by hindsight agents that can see future exogenous values")

    def select_action(self, state):
        # return action based on the current state
        raise NotImplementedError("This method should be implemented by subclasses")

### The dummy agent
The dummy agent is a simple baseline that always buys exactly the amount of energy needed to meet the demand, without using the battery at all.

In [ ]:
class DummyAgent(Agent):
    def select_action(self, state):
        return Action(buy_energy_amount=self.environment.demand_profile[state.time_step])
    

### The optimal in hindsight agent

The optimal in hindsight agent is a theoretical agent that has perfect foresight of the future market prices and demand, and can therefore make the optimal decision at each time step to minimize total costs.

It creates a MILP model of the entire time horizon, with the objective of minimizing total costs, and constraints representing the battery dynamics and feasibility.    

In [ ]:
def solve_MILP_model(exogen_data, parameters, demand_profile, value_of_battery_at_end_of_day=0):
    model = ConcreteModel()
    # Sets
    time_steps = range(len(exogen_data))
    model.T = Set(initialize=time_steps)

    # Parameters
    model.demand = Param(model.T, initialize={t: demand_profile[t] for t in model.T})
    model.energy_price = Param(model.T, initialize={t: exogen_data[t].energy_price for t in model.T})
    model.carbon_intensity = Param(model.T, initialize={t: exogen_data[t].carbon_intensity for t in model.T})

    model.max_battery_capacity = Param(initialize=parameters.max_battery_capacity)
    model.max_charging_rate = Param(initialize=parameters.max_charging_rate)
    model.max_discharging_rate = Param(initialize=parameters.max_discharging_rate)
    model.charge_efficiency = Param(initialize=parameters.charge_efficiency)
    model.discharge_efficiency = Param(initialize=parameters.discharge_efficiency)
    model.price_per_kg_co2 = Param(initialize=parameters.price_per_kg_co2)

    model.value_of_battery_at_end_of_day = Param(initialize=value_of_battery_at_end_of_day)


    # Variables
    model.buy_energy = Var(model.T, within=NonNegativeReals)
    model.charge_battery = Var(model.T, within=NonNegativeReals)
    model.discharge_battery = Var(model.T, within=NonNegativeReals)
    model.battery_level = Var(model.T, within=NonNegativeReals)
    model.charge = Var(model.T, within=Binary)  # 1 if charging, 0 if discharging

    # Objective
    def objective_rule(m):
        last_time_step = max(m.T)
        last_battery_level = m.battery_level[last_time_step] + m.charge_battery[last_time_step] * m.charge_efficiency - m.discharge_battery[last_time_step]
        return sum(
            m.buy_energy[t] * m.energy_price[t] + 
            m.price_per_kg_co2 * m.buy_energy[t] * m.carbon_intensity[t]
            for t in m.T
        ) - m.value_of_battery_at_end_of_day * last_battery_level
    model.objective = Objective(rule=objective_rule, sense=minimize)

    # Constraints
    # Demand must be met at each time step
    def demand_constraint_rule(m, t):
        return m.buy_energy[t] + m.discharge_battery[t] * m.discharge_efficiency == m.demand[t] + m.charge_battery[t]
    
    model.demand_constraint = Constraint(model.T, rule=demand_constraint_rule)

    # Battery level constraints
    def battery_level_rule(m, t):
        return m.battery_level[t] <= m.max_battery_capacity

    model.battery_capacity_constraint = Constraint(model.T, rule=battery_level_rule)

    def final_battery_level_rule(m):
        last_time_step = max(m.T)
        return m.battery_level[last_time_step] + m.charge_battery[last_time_step] * m.charge_efficiency - m.discharge_battery[last_time_step] <= m.max_battery_capacity
    
    model.final_battery_level_constraint = Constraint(rule=final_battery_level_rule)

    # Max charging and discharging rates
    def charging_rate_rule(m, t):
        return m.charge_battery[t] <= m.max_charging_rate * m.charge[t]
    model.charging_rate_constraint = Constraint(model.T, rule=charging_rate_rule)

    def discharging_rate_rule(m, t):
        return m.discharge_battery[t] <= m.max_discharging_rate * (1 - m.charge[t])
    model.discharging_rate_constraint = Constraint(model.T, rule=discharging_rate_rule)

    def discharging_less_than_battery_rule(m, t):
        return m.discharge_battery[t] <= m.battery_level[t]
    
    model.discharging_battery_constraint = Constraint(model.T, rule=discharging_less_than_battery_rule)

    # Battery level dynamics
    def battery_dynamics_rule(m, t):
        if t == 0:
            return m.battery_level[t] == parameters.initial_battery_level
        else:
            return m.battery_level[t] == m.battery_level[t-1] + m.charge_battery[t-1] * m.charge_efficiency - m.discharge_battery[t-1]
    model.battery_dynamics_constraint = Constraint(model.T, rule=battery_dynamics_rule)
    
    # Solve the model
    solver = SolverFactory('gurobi')
    if not solver.available():
        raise RuntimeError("Gurobi solver is not available in your environment.")
    results = solver.solve(model, tee=False)
    status = results.Solver()['Termination condition'].value
    assert status == 'optimal', f'error occurred, status: {status}.  Check model!'
    return model
    

In [ ]:
class OptimalHindsightAgent(Agent):
    def __init__(self, environment):
        super().__init__(environment)
        self.is_hindsight_policy = True
        self.expected_simulation_result = SimulationResult()

    def see_future_exogen_values(self, exogen_data):
        
        model = solve_MILP_model(exogen_data, self.environment.parameters, self.environment.demand_profile)

        # save optimal actions for each time step
        self.expected_simulation_result.states = [State(time_step=t,
                                                        battery_level=model.battery_level[t].value,
                                                        energy_price=model.energy_price[t],
                                                        carbon_intensity=model.carbon_intensity[t]) for t in model.T]
        self.expected_simulation_result.actions = [Action(buy_energy_amount=model.buy_energy[t].value) for t in model.T]
        self.expected_simulation_result.costs = [Cost(amount=model.buy_energy[t].value * model.energy_price[t],
                                                        carbon_emissions=model.buy_energy[t].value * model.carbon_intensity[t],
                                                        price_per_kg_co2=self.environment.parameters.price_per_kg_co2) for t in model.T]
            
    def select_action(self, state):
        return self.expected_simulation_result.actions[state.time_step]

### The optimize over the known future agent

At each day at 13:00, the market prices for the next day are revealed. This agent takes advantage of this information to optimize its actions for the next day.

In [ ]:
class OptimizeOverNextDayAgent(Agent):
    def __init__(self, environment):
        super().__init__(environment)
        #assert environment.parameters.price_per_kg_co2 == 0, "OptimizeOverNextDayAgent is only implemented for cost minimization (price_per_kg_co2=0)"
        self.expected_simulation_result = SimulationResult()
        self.previous_states = {}
        self.saved_exogen_data = {}

    def save_future_prices(self, state):
        self.previous_states[state.time_step] = state
        any_future_prices = False
        for t in state.other_factors['future_prices']:
            any_future_prices = True

            if t -24 < 0:
                carbon_intensity = state.carbon_intensity
            else:
                carbon_intensity = self.saved_exogen_data[t - 24].carbon_intensity

            self.saved_exogen_data[t] = ExogenousValues(
                energy_price=state.other_factors['future_prices'][t],
                carbon_intensity=carbon_intensity,  # Carbon intensity is not relevant for this agent since price_per_kg_co2=0
                other_factors={}
            )
        return any_future_prices

    def get_exogen_data(self, state):
        exogen_data = []
        t = state.time_step
        while t in self.saved_exogen_data:
            exogen_data.append(self.saved_exogen_data[t])
            t += 1
        return exogen_data

    def update_expected_simulation_result(self, state, exogen_data, value_of_battery_at_end_of_day=0):
        parameters = deepcopy(self.environment.parameters)
        parameters.initial_battery_level = state.battery_level


        lookahead_horizon = min(len(exogen_data), len(self.environment.demand_profile) - state.time_step)

        exogen_data = exogen_data[:lookahead_horizon]        
        demand_profile = self.environment.demand_profile[state.time_step:state.time_step + lookahead_horizon]

        model = solve_MILP_model(exogen_data, parameters, demand_profile, value_of_battery_at_end_of_day=value_of_battery_at_end_of_day)
        
        # save states, actions, and costs for the optimized period
        for t in range(state.time_step, state.time_step + lookahead_horizon):
            new_state = State(time_step=t,
                            battery_level=model.battery_level[t - state.time_step].value,
                            energy_price=model.energy_price[t - state.time_step],
                            carbon_intensity=model.carbon_intensity[t - state.time_step])
            action = Action(buy_energy_amount=model.buy_energy[t - state.time_step].value)
            cost = Cost(amount=model.buy_energy[t - state.time_step].value * model.energy_price[t - state.time_step],
                        carbon_emissions=model.buy_energy[t - state.time_step].value * model.carbon_intensity[t - state.time_step],
                        price_per_kg_co2=self.environment.parameters.price_per_kg_co2)
            if t < len(self.expected_simulation_result.states):
                self.expected_simulation_result.states[t] = new_state
                self.expected_simulation_result.actions[t] = action
                self.expected_simulation_result.costs[t] = cost
            else:
                self.expected_simulation_result.states.append(new_state)
                self.expected_simulation_result.actions.append(action)
                self.expected_simulation_result.costs.append(cost)


    def select_action(self, state):
        
        any_future_prices = self.save_future_prices(state)

        if any_future_prices:
            exogen_data = self.get_exogen_data(state)
            self.update_expected_simulation_result(state, exogen_data)

        return self.expected_simulation_result.actions[state.time_step]
        
    

### Adding a value to the final battery level

The optimize over next day agent can be modified to include a value for the final battery level, which encourages the agent to not only minimize costs for the next day, but also to consider the long-term value of having energy stored in the battery for future use.


In [ ]:
# Optimize over next day with value for battery level at the end of the day
# how to determine value of energy at the end of the day? just moving average of energy prices?

class OptimizeOverNextDayWithBatteryValueAgent(OptimizeOverNextDayAgent):

    def calculate_value_of_battery_at_end_of_day(self, exogen_data):
        # This is a placeholder implementation. In a real implementation, you would want to calculate this based on the expected future energy prices and demand, as well as the cost of charging/discharging the battery.
        previous_prices = [data.energy_price for data in exogen_data]
        previous_carbon_intensities = [data.carbon_intensity for data in exogen_data]
        return self.environment.parameters.discharge_efficiency * (np.mean(previous_prices) + self.environment.parameters.price_per_kg_co2 * np.mean(previous_carbon_intensities))


    def select_action(self, state):
        any_future_prices = self.save_future_prices(state)

        if any_future_prices:
            exogen_data = self.get_exogen_data(state)
            value_of_battery_at_end_of_day = self.calculate_value_of_battery_at_end_of_day(exogen_data)
            self.update_expected_simulation_result(state, exogen_data, value_of_battery_at_end_of_day)  # Example value for battery at the end of the day, this could be determined more systematically

        return self.expected_simulation_result.actions[state.time_step]
        

### Our agent that uses prediction for future market prices and carbon intensity

This agent uses the LSTM models to predict future market prices and carbon intensity, and then optimizes its actions based on these predictions.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 48, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(48, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

class LSTMModelCarbon(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 48, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(48, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

class LSTMModelPrice(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 48, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(48, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("lstm_model_carbon.pkl", "rb") as f:
    carboncheckpoint = pickle.load(f)

carbonmodel = carboncheckpoint["model"]

with open("lstm_model_price.pkl", "rb") as f:
    pricecheckpoint = pickle.load(f)

pricemodel = pricecheckpoint["model"]





In [ ]:
print(carbonmodel.state_dict())


In [ ]:
print(carboncheckpoint['model_state'])

In [ ]:
class AgentThatUsesLSTM(OptimizeOverNextDayWithBatteryValueAgent):
    def __init__(self, environment):
        super().__init__(environment)
        self.lstmPriceModel = pricemodel.to(device)
        self.lstmCarbonModel = carbonmodel.to(device)  # Not implemented yet, but could be added similarly to the price model

        self.saved_predicted_exogen_values = {}

    def predict_future_exogenous_values(self, state):
        with torch.no_grad():
            pricePrediction = self.lstmPriceModel(state.other_factors["lstmPriceInput"]).squeeze(0).cpu().numpy()
            carbonPrediction = self.lstmCarbonModel(state.other_factors["lstmCarbonInput"]).squeeze(0).cpu().numpy()

        #print(f"Predicted future prices: {pricePrediction}")
        #print(f"Predicted future carbon intensities: {carbonPrediction}")

        # Add to saved predicted exogenous values
        lookahead_horizon = min(len(pricePrediction), len(carbonPrediction))

        for t in range(state.time_step +1, state.time_step + 1 + lookahead_horizon):
            self.saved_predicted_exogen_values[t] = ExogenousValues(
                    energy_price=pricePrediction[t - (state.time_step + 1)]/1000,  # Assuming the model predicts price in EUR/MWh and we need EUR/kWh
                    carbon_intensity=carbonPrediction[t - (state.time_step + 1)]/1000,  # Carbon intensity is not relevant for this agent since price_per_kg_co2=0
                    other_factors={}
                )



    def get_exogen_data(self, state):
        exogen_data = [ExogenousValues(energy_price=state.energy_price, carbon_intensity=state.carbon_intensity)]
        t = state.time_step + 1
        while t in self.saved_exogen_data:
            exogen_data.append(ExogenousValues(
                energy_price=self.saved_exogen_data[t].energy_price,  # Assuming the model predicts price in EUR/MWh and we need EUR/kWh
                carbon_intensity=self.saved_predicted_exogen_values[t].carbon_intensity) 
                )
            t += 1
        
        while t in self.saved_predicted_exogen_values:
            exogen_data.append(self.saved_predicted_exogen_values[t])
            t += 1

        return exogen_data


    def select_action(self, state):
        print(f"Selecting action for time step {state.time_step}")#, battery level {state.battery_level}, energy price {state.energy_price}, carbon intensity {state.carbon_intensity}")
        self.predict_future_exogenous_values(state)
        self.save_future_prices(state)
                
        
        exogen_data = self.get_exogen_data(state)
        #print([data.energy_price for data in exogen_data])
        #print([data.carbon_intensity for data in exogen_data])
        value_of_battery_at_end_of_day = self.calculate_value_of_battery_at_end_of_day(exogen_data)
        self.update_expected_simulation_result(state, exogen_data, value_of_battery_at_end_of_day)  # Example value for battery at the end of the day, this could be determined more systematically
        #print(f"Optimal actions: {[(f'Time step {t}: Action {action.buy_energy_amount}') for t, action in enumerate(self.expected_simulation_result.actions)]}")

        return self.expected_simulation_result.actions[state.time_step]



# Loading the exogenous variables

## Loading the demand data

Here we load the demand data to be used in the simulation. 


In [ ]:
# demand profile

demand_df = pd.read_csv("steel_industry_hourly_usage.csv")


# get weekday from date column and add as new column
demand_df['date'] = pd.to_datetime(demand_df['date'])
demand_df['week'] = demand_df['date'].dt.isocalendar().week
demand_df['day_of_week'] = demand_df['date'].dt.weekday
demand_df['hour_of_day'] = demand_df['date'].dt.hour

# remove duplicate rows in week and day of week and hour of day
demand_df = demand_df.drop_duplicates(subset=['week', 'day_of_week', 'hour_of_day'])

# print sum of demand
print(f"Total demand in dataset: {demand_df['Usage_kWh'].sum()} kWh")

print(f"Average hourly demand: {demand_df['Usage_kWh'].mean()} kWh")

print(f"Average daily demand: {demand_df.groupby(['week', 'day_of_week'])['Usage_kWh'].sum().mean()} kWh")


## Loading the market price and carbon intensity data

Here we load the market price and carbon intensity data to be used in the simulation.

The demand data is for 2018, but our other data is for 2024.
To align the demand data with our other data, we can assume that the demand patterns are similar across years, so we join the data based on week, weekday, and hour.

In [ ]:
df = pd.read_csv("final_data.csv")

df['datetime'] = pd.to_datetime(df['datetime'])

df = df[df['datetime'] >= '2024-01-01']

df = df[df['zone'] == 'DK-DK1']

df['week'] = df['datetime'].dt.isocalendar().week

# merge df with demand df on week, day_of_week and hour_of_day
print(len(df))
df = pd.merge(df, demand_df, how='left', on=['week', 'day_of_week', 'hour_of_day'])
print(len(df))

print(df[df['Usage_kWh'].isnull()])


## Loading the lstm model input as tensors

The lstm model input is scaled and transformed into tensors similar to how it was done during training, so that it can be used for making predictions in the simulation environment.

Here we load the input data for the LSTM models to be used in the simulation.

In [ ]:
with open("exportDataPrice.pkl", "rb") as f:
    priceData = pickle.load(f)

X_price = priceData[0] 
X_tensor_price = torch.tensor(X_price , dtype=torch.float32).to(device)

print(X_tensor_price.shape)
print(X_tensor_price[0])
print(priceData[1].shape)
print(priceData[1][0])
print(priceData[1][-1])

In [ ]:
with open("exportDataCarbon.pkl", "rb") as f:
    carbonData = pickle.load(f)

X_carbon = carbonData[0] 
X_tensor_carbon = torch.tensor(X_carbon , dtype=torch.float32).to(device)

print(X_tensor_carbon.shape)
print(X_tensor_carbon[0])
print(carbonData[1].shape)
print(carbonData[1][0])
print(carbonData[1][-1])

## Creating the trajectories

Each hour new market information is revealed (e.g., weather conditions, economic indicators)

Additionally, each day at 13:00 the market prices for the next day are revealed, which allows the agent to plan its actions for the next day.

Here we combine all the loaded data into trajectories that can be used in the simulation environment. 


In [ ]:

all_exogen_trajectories = []


#zones = df['zone'].unique()
zones = ['DK-DK1']

for zone in zones:
    zone = 'DK-DK1'
    #zone_data = df[df['zone'] == zone]

    # add index column to zone data
    #zone_data = zone_data.sort_values('datetime').reset_index(drop=True)
    zone_data = df.sort_values('datetime').reset_index(drop=True)

    zone_exogen_data = []
    zone_demands = []

    for i, row in zone_data.iterrows():
        future_prices = {}

        row_hour = row['hour_of_day']

        if i == 0:
            # add prices for this day
            day_data = zone_data[zone_data['datetime'].dt.date == row['datetime'].date()]
            future_prices = {j: future_row['value_spot']/1000 for j, future_row in day_data.iterrows()}
            if row_hour >= 13: # also add prices for next day¨
                next_day_data = zone_data[zone_data['datetime'].dt.date == row['datetime'].date() + pd.Timedelta(days=1)]
                future_prices.update({j: future_row['value_spot']/1000 for j, future_row in next_day_data.iterrows()})

        
        elif row_hour == 13: # if its 1 pm or after also add prices for the next day     
            next_day_data = zone_data[zone_data['datetime'].dt.date == row['datetime'].date() + pd.Timedelta(days=1)]
            future_prices = {j: future_row['value_spot']/1000 for j, future_row in next_day_data.iterrows()}

        zone_exogen_data.append(ExogenousValues(
            energy_price=row['value_spot']/1000,  # Convert to EUR/kWh
            carbon_intensity=row['carbonIntensity']/1000,  # Convert to kgCO2/kWh
            other_factors={'future_prices': future_prices, 
                           'lstmPriceInput': X_tensor_price[i].unsqueeze(0),
                           'lstmCarbonInput': X_tensor_carbon[i].unsqueeze(0)
                           }
        ))
        zone_demands.append(row['Usage_kWh'])
    
    all_exogen_trajectories.append(ExogenousTrajectory(zone, zone_exogen_data, zone_demands))


## The battery parameters

Here we define the parameters of the battery to be used in the simulation.



In [ ]:
parameters = Parameters(
    initial_battery_level=0, # Initial battery level in kWh
    max_battery_capacity=1000, # Maximum battery capacity in kWh
    max_charging_rate=1000, # Maximum charge rate in kW
    charge_efficiency=0.98, # Efficiency of charging (0 to 1)
    max_discharging_rate=1000, # Maximum discharge rate in kW
    discharge_efficiency=0.98, # Efficiency of discharging (0 to 1)
    price_per_kg_co2=0.0 # Price per kg of CO2 emissions (currency per kg)
)

# Running the simulation

Now we run the simulation for each of the agents and compare their performance in terms of total costs and carbon emissions.

In [ ]:
environment = Environment(parameters)

policies = []

policies.append(DummyAgent)
policies.append(OptimizeOverNextDayAgent)
policies.append(OptimizeOverNextDayWithBatteryValueAgent)
policies.append(AgentThatUsesLSTM)
policies.append(OptimalHindsightAgent)

all_results = environment.evaluate_policies(policies, all_exogen_trajectories)

Plotting the values for the first week of the simulation for each agent to see how the battery level, market prices, and demand evolve over time.

In [ ]:
start = 0

end = 24 * 14

for result in all_results:
    print(f"Policy: {result.policy_name}")
    result.print_average_costs()
    for sim in result.simulation_results:
        sim.plot(environment, start=start, end=end)
        


Here we make a table showing the each type of cost for each agent.

In [ ]:
# print table for each cost where rows are policies and columns are zones

for cost_type in ['amount', 'carbon_emissions', 'total_cost']:
    print(f"Average {cost_type} for each policy and zone:")
    df = pd.DataFrame(columns=['Policy'] + list(zones))
    for result in all_results:
        row = {'Policy': result.policy_name}
        for sim in result.simulation_results:
            row[sim.exogen_trajectory.zone] = sim.get_total_cost(cost_type)
        df.loc[len(df)] = row
    print(df.to_string(index=False))

Here we compare the total costs compared to the dummy agent for each agent.

In [ ]:
# Economic savings compared to dummy agent
dummy_results = next(result for result in all_results if result.policy_name == 'DummyAgent').simulation_results
for result in all_results:
    if result.policy_name == 'DummyAgent':
        continue
    print(f"Economic savings for {result.policy_name} compared to DummyAgent:")
    df = pd.DataFrame(columns=['Zone', 'Dummy Cost', 'Cost','Total Savings', 'Savings per year'])
    for sim, dummy_sim in zip(result.simulation_results, dummy_results):
        dummy_cost = dummy_sim.get_total_cost('total_cost')
        policy_cost = sim.get_total_cost('total_cost')
        savings = dummy_cost - policy_cost
        savings_per_year = savings/ (len(sim.states) / (24 * 365))  # Annualize savings based on the number of time steps in the simulation
        df.loc[len(df)] = [sim.exogen_trajectory.zone, dummy_cost, policy_cost, savings, savings_per_year]
    print(df.to_string(index=False))

## Cost saving per kWh compared to the dummy agent

Here we iterate over different battery sizes and different agent configurations to see how the cost savings per kWh compared to the dummy agent change with different battery sizes and agent strategies.

In [ ]:
battery_sizes = [1, 5, 10, 50, 100, 500, 1000, 5000, 10000]
policies = [DummyAgent, OptimizeOverNextDayAgent, OptimizeOverNextDayWithBatteryValueAgent, OptimalHindsightAgent]

all_results_by_battery_size = {}



for battery_size in battery_sizes:
    parameters = Parameters(
        initial_battery_level=0, # Initial battery level in kWh
        max_battery_capacity=battery_size, # Maximum battery capacity in kWh
        max_charging_rate=battery_size, # Maximum charge rate in kW
        charge_efficiency=0.98, # Efficiency of charging (0 to 1)
        max_discharging_rate=battery_size, # Maximum discharge rate in kW
        discharge_efficiency=0.98, # Efficiency of discharging (0 to 1)
        price_per_kg_co2=0.1 # Price per kg of CO2 emissions (currency per kg)
    )
    environment = Environment(parameters)
    all_results = environment.evaluate_policies(policies, all_exogen_trajectories)
    all_results_by_battery_size[battery_size] = all_results
   

Then we save the annual costs, annual savings, cost saving per kWh, and net present value of the savings for each agent and battery size for further analysis.

For the net present value calculation, we assume a discount rate of 5% and a battery lifespan of 15 years.
 

In [ ]:
battery_lifetime = 15
market_rate = 0.05
battery_cost_per_kwh = 200  # Example cost of battery per kWh, this could be varied or calculated more systematically based on real data

annual_costs_results = {}
annual_savings_results = {}
savings_per_kwh_results = {}
net_present_value_per_kwh_results = {}
total_net_present_value_results = {}
total_net_present_profit_results = {}

for battery_size, all_results in all_results_by_battery_size.items():
    dummy_results = next(result for result in all_results if result.policy_name == 'DummyAgent').simulation_results
    
    for result in all_results:
        policy = result.policy_name
        for sim, dummy_sim in zip(result.simulation_results, dummy_results):
            zone = sim.exogen_trajectory.zone

            dummy_cost = dummy_sim.get_total_cost('total_cost')
            policy_cost = sim.get_total_cost('total_cost')
            annual_costs_results[(policy, zone, battery_size)] = policy_cost / (len(sim.states) / (24 * 365))  # Annualize cost based on the number of time steps in the simulation
            savings = dummy_cost - policy_cost
            savings_per_year = savings/ (len(sim.states) / (24 * 365))
            annual_savings_results[(policy, zone, battery_size)] = savings_per_year

            savings_per_kwh = savings / battery_size
            savings_per_kwh_results[(policy, zone, battery_size)] = savings_per_kwh

            net_present_value_per_kwh = sum(savings_per_kwh / (1 + market_rate) ** year for year in range(1, battery_lifetime + 1))
            net_present_value_per_kwh_results[(policy, zone, battery_size)] = net_present_value_per_kwh

            total_net_present_value = net_present_value_per_kwh * battery_size
            total_net_present_value_results[(policy, zone, battery_size)] = total_net_present_value

            total_net_present_profit = total_net_present_value - (battery_cost_per_kwh * battery_size)
            total_net_present_profit_results[(policy, zone, battery_size)] = total_net_present_profit
            
    


Here we plot the annual costs, annual savings, cost saving per kWh, and net present value of the savings for each agent and battery size to visualize the results of our analysis.

In [ ]:
# plot savings per year for each policy and battery size
zone = 'DK-DK1'

what_to_plot = {"Annual Costs": annual_costs_results, "Annual Savings": annual_savings_results, "Savings per kWh per Year": savings_per_kwh_results, "Net Present Value per kWh": net_present_value_per_kwh_results, "Total Net Present Value": total_net_present_value_results, "Total Net Present Profit": total_net_present_profit_results}


for title, results in what_to_plot.items():
        
    plt.figure(figsize=(12, 8))
    for policy in policies:
        annual = [results.get((policy.__name__, zone, battery_size), 0) for battery_size in battery_sizes]
        plt.plot(battery_sizes, annual, label=policy.__name__)

    plt.xlabel('Battery Size (kWh)')
    plt.ylabel(f'{title} (EUR)')
    plt.title(f'{title} by Battery Size for Zone {zone}')
    plt.legend()
    plt.grid()
    plt.show()



# Conclusion

With our model it is easy to compare the different policies and different battery parameters to see how they affect the cost savings for the consumer.

This means that if we know how much the initial investment of buying a battery requires per kWh, then we can easily determine how big the battery needs to be for the cost savings to outweigh the initial investment, and how much the consumer can expect to save each year of the battery's life.

As an example, for the client which we have used demand data for and assuming an initial investment of 200 EUR per kWh, we can see that a battery size of less than 1000 kWh would be cost-effective. If the battery size is larger than this, the cost savings would not outweigh the initial investment, and it would not be a financially viable option for the consumer.  

This analysis could also be done for different initial investment costs, different battery lifespans, different discount rates, different demand patterns, different battery parameters, and different years of data to see how these factors affect the cost-effectiveness of the battery storage system.

We have assumed 2024 market prices to be representative of future market prices, but it is important to note that market prices can be volatile and may not always follow historical patterns. Therefore, it is crucial to consider the uncertainty in future market prices when making decisions about investing in battery storage systems.

We have also assumed that future demand is known. For many consumers, this is somewhat reasonable, as much of energy consumption is predictable (e.g., heating, cooling, appliances). However, for some consumers with more variable consumption patterns, this assumption may not hold, and it would be important to consider the uncertainty in future demand when making decisions about investing in battery storage systems. This is something that could be explored in future work by modeling demand as a stochastic process or by using historical data to estimate the distribution of future demand.

Other things that could be included in the model are:

Including the option to sell excess energy back to the market, this would add another dimension to the action space and could potentially increase the cost savings for the consumer.

Adding local energy generation (e.g., solar panels or wind turbines) to the model, which would allow the consumer to generate their own energy and potentially reduce their reliance on the market.

Modeling the degradation of the battery over time, as this would affect the long-term cost-effectiveness of the battery storage system.

Modeling passive energy losses, as the battery may lose energy over time even when not in use, which would affect the overall efficiency of the system.

Adding non-linearities to the charging and discharging processes, as many batteries have non-linear charging and discharging characteristics that could affect the optimal strategy for using the battery.